# Spark GraphFrames

In [5]:
!pip cache purge --quiet

In [6]:
!pip install install-jdk==1.1.0 \
             graphframes-py==0.9.2 \
             pyspark==4.0.0 --quiet

In [7]:
import jdk
import os

from graphframes import GraphFrame
from pyspark.sql import SparkSession, functions as F
from singlestoredb.management import get_secret

os.environ.pop("CONTAINER_ID", None)

'727638ba-add0-461b-b510-2a3a16b0b7b0'

In [8]:
# Configure Java
jdk_path = jdk.install("17")

os.environ["JAVA_HOME"] = jdk_path
os.environ["PATH"] = f"{jdk_path}/bin:" + os.environ["PATH"]

In [9]:
# Define fully-qualified Maven coordinates for JARs
jar_packages = [
    "com.singlestore:singlestore-spark-connector_2.13:4.2.0-spark-4.0.0",
    "com.singlestore:singlestore-jdbc-client:1.2.5",
    "org.apache.commons:commons-dbcp2:2.12.0",
    "org.apache.commons:commons-pool2:2.12.0",
    "io.spray:spray-json_2.13:1.3.6",
    "io.graphframes:graphframes-spark4_2.13:0.9.2"
]

spark = (
    SparkSession.builder
    .appName("Spark GraphFrames")
    .master("local[*]")
    .config("spark.jars.packages", ",".join(jar_packages))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
com.singlestore#singlestore-spark-connector_2.13 added as a dependency
com.singlestore#singlestore-jdbc-client added as a dependency
org.apache.commons#commons-dbcp2 added as a dependency
org.apache.commons#commons-pool2 added as a dependency
io.spray#spray-json_2.13 added as a dependency
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7319ae03-82c3-4ff2-a19c-1d62b8160d10;1.0
	confs: [default]
	found com.singlestore#singlestore-spark-connector_2.13;4.2.0-spark-4.0.0 in central
	found org.apache.avro#avro;1.11.3 in central
	found com.fasterxml.jackson.core#jackson-core;2.14.2 in central
	found com.fasterxml.jackson.core#jackson-databind;2.14.2

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [11]:
from sqlalchemy import *

db_connection = create_engine(connection_url)
url = db_connection.url

In [12]:
password = get_secret("password")
database = url.database
host = url.host
port = url.port
cluster = host + ":" + str(port)

In [13]:
spark.conf.set("spark.datasource.singlestore.ddlEndpoint", cluster)
spark.conf.set("spark.datasource.singlestore.user", "admin")
spark.conf.set("spark.datasource.singlestore.password", password)
spark.conf.set("spark.datasource.singlestore.disablePushdown", "false")

In [14]:
connections = (spark.read
    .format("singlestore")
    .load(f"{database}.london_connections")
)

In [15]:
stations = (spark.read
    .format("singlestore")
    .load(f"{database}.london_stations")
)

In [16]:
underground = GraphFrame(stations, connections)

In [17]:
(underground
    .vertices
    .show(5, truncate = False)
)

+---------------------------------------------------+------------------+-------------------+-------+
|id                                                 |latitude          |longitude          |zone   |
+---------------------------------------------------+------------------+-------------------+-------+
|Wood Green                                         |51.59745355165763 |-0.1095265887761309|3      |
|Lloyd Park                                         |51.36427540535568 |-0.0807462726980878|3,4,5,6|
|Putney Bridge                                      |51.46786499705636 |-0.2093654143019544|2      |
|London Bridge                                      |51.50467420930433 |-0.0860055977481396|1      |
|Edgware Road (Circle/District/Hammersmith and City)|51.519997771093394|-0.1676682526511785|1      |
+---------------------------------------------------+------------------+-------------------+-------+
only showing top 5 rows


In [19]:
(underground
    .edges
    .show(5)
)

+---------+-------------+--------------------+
|tube_line|          src|                 dst|
+---------+-------------+--------------------+
| District|Cannon Street|       Mansion House|
| Northern|Woodside Park|Totteridge and Wh...|
| Tramlink|  Merton Park|         Morden Road|
|  Central|   West Acton|     Ealing Broadway|
| District|Parsons Green|       Putney Bridge|
+---------+-------------+--------------------+
only showing top 5 rows


In [20]:
# Count how many stations are in each zone
(underground
    .vertices
    .groupBy("zone")
    .count()
    .orderBy("count", ascending = False)
    .show()
)

+-------+-----+
|   zone|count|
+-------+-----+
|      2|   75|
|      1|   62|
|      3|   55|
|      4|   49|
|3,4,5,6|   32|
|      5|   24|
|      6|   19|
|    2,3|   14|
|    3,4|    6|
|    1,2|    4|
|      7|    4|
|      9|    2|
|    5,6|    1|
|      8|    1|
|    6,7|    1|
+-------+-----+



In [21]:
# Find the number of stations by the line name (e.g. District)
(underground
    .edges
    .filter("tube_line = 'District'")
    .count()
)

59

In [22]:
# List all stations that appear on a specific line (e.g. Circle)
(underground
    .edges
    .filter("tube_line = 'Circle'")
    .select("src")
    .union(
        underground
            .edges
            .filter("tube_line = 'Circle'")
            .select("dst")
        )
     .distinct()
     .show(100, truncate = False)
)

+---------------------------------------------------+
|src                                                |
+---------------------------------------------------+
|Gloucester Road                                    |
|Blackfriars                                        |
|Wood Lane                                          |
|Ladbroke Grove                                     |
|Sloane Square                                      |
|Temple                                             |
|Tower Hill                                         |
|Latimer Road                                       |
|Royal Oak                                          |
|Great Portland Street                              |
|Westminster                                        |
|Westbourne Park                                    |
|Notting Hill Gate                                  |
|Farringdon                                         |
|Bayswater                                          |
|Goldhawk Road              

In [23]:
# Find all lines passing through a given station (e.g. Paddington)
(underground
    .edges
    .filter("src = 'Paddington' OR dst = 'Paddington'")
    .select("tube_line")
    .distinct()
    .show()
)

+--------------------+
|           tube_line|
+--------------------+
|              Circle|
|            District|
|            Bakerloo|
|Hammersmith and City|
+--------------------+



In [24]:
# Show top 5 stations with the most connections
(underground
    .edges
    .groupBy("src")
    .count()
    .orderBy(F.desc("count"))
    .show(5, truncate = False)
)

+-----------------------+-----+
|src                    |count|
+-----------------------+-----+
|Kings Cross St. Pancras|6    |
|Earls Court            |5    |
|Baker Street           |5    |
|Embankment             |4    |
|West Ham               |4    |
+-----------------------+-----+
only showing top 5 rows


In [25]:
spark.stop()